In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from io import BytesIO
from minio import Minio

In [ ]:
client = Minio("localhost:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)
pid = "CGMacros-012" # Validando um paciente por vez
gold_path = f"gold/CGMacros/{pid}/{pid}_ML_READY.parquet"

In [ ]:
# 1. Carregar os dados da Gold
response = client.get_object("cgmacros", gold_path)
df_gold = pd.read_parquet(BytesIO(response.read()))
response.close()

# 2. Criar Painel de Validação
fig, axes = plt.subplots(3, 1, figsize=(15, 18))

# --- Gráfico 1: Validação do Target (O "Pulo no Tempo") ---
# Vamos pegar uma janela de 6 horas para ver o detalhe
sample = df_gold.iloc[100:300]
axes[0].plot(sample['Timestamp'], sample['Libre GL'], label='Glicose Atual', marker='o', alpha=0.6)
axes[0].plot(sample['Timestamp'], sample['target_glucose'], label='Target (Previsão +30min)', linestyle='--', color='red')
axes[0].set_title(f"Validação de Deslocamento Temporal (Target vs Atual) - {pid}")
axes[0].legend()
# Dica: A linha vermelha deve ser uma cópia da azul, mas "empurrada" para a esquerda.

# --- Gráfico 2: Validação da Janela de Alimentação ---
sns.lineplot(ax=axes[1], data=df_gold.iloc[:1000], x='Timestamp', y='Libre GL', color='gray', alpha=0.5)
# Pintar o fundo onde In_Meal_Window é 1
axes[1].fill_between(df_gold.iloc[:1000]['Timestamp'], 50, 350,
                 where=df_gold.iloc[:1000]['In_Meal_Window'] == 1,
                 color='orange', alpha=0.2, label='Janela Pós-Prandial (2h)')
axes[1].set_title("Alinhamento: Picos de Glicose vs Janelas de Alimentação")
axes[1].legend()

# --- Gráfico 3: Correlação de Features (Lags vs Target) ---
cols_to_corr = ['target_glucose', 'Libre GL', 'lag_GL_5min', 'lag_GL_15min', 'lag_GL_30min', 'diff_5min', 'HR']
corr = df_gold[cols_to_corr].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[2])
axes[2].set_title("Matriz de Correlação: Quais features mais explicam o Target?")

plt.tight_layout()
plt.show()

In [ ]:
print(df_gold[['Timestamp', 'Libre GL', 'target_glucose']].head(10))

In [ ]:
import pandas as pd
from io import BytesIO
from minio import Minio

# Configurações do seu ambiente local
MINIO_ENDPOINT = "localhost:9000"
MINIO_BUCKET = "cgmacros"
# O prefixo onde os metadados foram salvos
PREFIX = "gold/metadata/"

client = Minio(
    MINIO_ENDPOINT,
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False,
)

try:
    # 1. Listar os objetos dentro da pasta de metadados
    objects = client.list_objects(MINIO_BUCKET, prefix=PREFIX, recursive=True)

    csv_found = False
    for obj in objects:
        # 2. Verificar se o arquivo é um CSV
        if obj.object_name.endswith('.csv'):
            csv_found = True
            print(f"\n📄 Lendo arquivo: {obj.object_name}")
            print("-" * 50)

            # 3. Buscar o conteúdo do arquivo
            response = client.get_object(MINIO_BUCKET, obj.object_name)

            try:
                # 4. Carregar no Pandas e printar as primeiras 10 linhas
                df_stats = pd.read_csv(BytesIO(response.read()))
                print(df_stats.head(10))
                print("-" * 50)
            finally:
                response.close()
                response.release_conn()

    if not csv_found:
        print(f"Nenhum arquivo .csv encontrado em {MINIO_BUCKET}/{PREFIX}")

except Exception as err:
    print(f"Erro ao acessar o Minio: {err}")

## CGMacros-012

| statistic | Timestamp | Libre GL | Dexcom GL | HR | Calories (Activity) | METs | Calories | Carbs | Protein |
|-----------|-----------|----------|-----------|----|---------------------|------|----------|-------|---------|
| count | 14565 | 14565.00 | 14270.00 | 13321.00 | 14565.00 | 14565.00 | 33.00 | 33.00 | 33.00 |
| mean | 2023-03-03 11:05:05 | 164.65 | 139.98 | 80.03 | 1.98 | 16.08 | 462.03 | 46.85 | 30.48 |
| min | 2023-02-26 09:43:00 | 77.00 | 72.00 | 59.00 | 1.23 | 10.00 | 0.00 | 0.00 | 0.00 |
| 25% | 2023-02-28 22:24:00 | 132.00 | 113.00 | 74.00 | 1.23 | 10.00 | 170.00 | 19.00 | 12.00 |
| 50% | 2023-03-03 11:05:00 | 157.00 | 135.80 | 80.00 | 1.48 | 12.00 | 448.00 | 43.00 | 22.00 |
| 75% | 2023-03-05 23:46:00 | 185.67 | 160.00 | 84.00 | 1.60 | 13.00 | 608.00 | 72.00 | 44.00 |
| max | 2023-03-08 12:42:00 | 342.00 | 300.00 | 148.00 | 12.80 | 104.00 | 1180.00 | 94.00 | 88.00 |
| std | NaN | 49.36 | 37.75 | 8.53 | 1.42 | 11.50 | 305.84 | 30.48 | 26.56 |

---

## CGMacros-039

| statistic | Timestamp | Libre GL | Dexcom GL | HR | Calories (Activity) | METs | Calories | Carbs | Protein |
|-----------|-----------|----------|-----------|----|---------------------|------|----------|-------|---------|
| count | 17115 | 17115.00 | 14275.00 | 14330.00 | 17115.00 | 17115.00 | 34.00 | 34.00 | 34.00 |
| mean | 2025-02-13 09:05:00 | 139.06 | 209.97 | 79.90 | 1.31 | 14.89 | 490.82 | 52.38 | 29.32 |
| min | 2025-02-07 10:28:00 | 59.00 | 101.00 | 56.00 | 0.88 | 10.00 | 0.00 | 0.00 | 0.00 |
| 25% | 2025-02-10 09:46:30 | 109.17 | 171.60 | 70.00 | 0.88 | 10.00 | 268.00 | 25.75 | 3.75 |
| 50% | 2025-02-13 09:05:00 | 126.13 | 193.00 | 77.00 | 0.88 | 10.00 | 446.50 | 62.00 | 22.00 |
| 75% | 2025-02-16 08:23:30 | 160.00 | 232.60 | 86.00 | 1.06 | 12.00 | 686.00 | 73.00 | 43.25 |
| max | 2025-02-19 07:42:00 | 321.00 | 400.00 | 169.00 | 9.53 | 108.00 | 1331.00 | 94.00 | 96.00 |
| std | NaN | 44.60 | 59.04 | 14.83 | 0.97 | 10.94 | 336.52 | 28.98 | 29.21 |

---

Colunas de lags e features temporais (`lag_*`, `diff_*`, `hour_*`) foram omitidas por serem derivadas — incluir se necessário.
